# Graph Attention Networks (GAT) — Colab

This notebook mirrors `code/model.py`, `code/train.py`, and `code/evaluate.py`: **2-layer GAT** on **Cora** / **CiteSeer**, 100 runs, **mean ± std** test accuracy (paper Table 2: **83.0 ± 0.7%** / **72.5 ± 0.7%**).

Training matches `train.py`: early stopping on **validation loss**, **best checkpoint restored**, **test accuracy computed once** after training (no peeking at test during the inner loop).

## Colab setup

1. **Runtime → Change runtime type →** GPU (recommended).
2. **Optional but recommended:** mount Google Drive so Planetoid data and CSVs persist. The config cell auto-uses  
   `/content/drive/MyDrive/[Cornell] Spring Junior/CS 4782/gat-reimplementation/{gat_data,results}` when that folder exists.
3. Run **Install** once per session.
4. Set `NUM_RUNS` in the last cell (`3` smoke test, `100` for paper protocol).

In [1]:
# @title Install dependencies (run once per Colab session)
!pip install -q torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 26.1 MB/s eta 0:00:00


In [2]:
# @title Config: paths and imports
import csv
import pathlib
import sys
from typing import List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GATConv
from torch_geometric.transforms import NormalizeFeatures

# If Drive is mounted at the course repo path, data + CSVs persist there.
_DRIVE_REPO = pathlib.Path(
    "/content/drive/MyDrive/[Cornell] Spring Junior/CS 4782/gat-reimplementation"
)
if _DRIVE_REPO.exists():
    PLANETOID_PARENT = _DRIVE_REPO / "gat_data"
    RESULTS_DIR = _DRIVE_REPO / "results"
else:
    PLANETOID_PARENT = pathlib.Path("/content/gat_data")
    RESULTS_DIR = pathlib.Path("/content/gat_results")

PLANETOID_PARENT.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(), "| cwd:", pathlib.Path().resolve())
print("data dir:", PLANETOID_PARENT)
print("results:", RESULTS_DIR)

torch: 2.10.0+cu128 | cuda: True | cwd: /content


In [3]:
# @title Model (same as `code/model.py`)
class GAT(nn.Module):
    def __init__(self, num_features: int, num_classes: int, dropout: float = 0.6):
        super().__init__()
        self.dropout = dropout

        self.conv1 = GATConv(
            in_channels=num_features,
            out_channels=8,
            heads=8,
            dropout=dropout,
            concat=True,
        )
        self.conv2 = GATConv(
            in_channels=8 * 8,
            out_channels=num_classes,
            heads=1,
            dropout=dropout,
            concat=False,
        )

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)

In [4]:
# @title Training (same as `code/train.py`)
import copy
import random

LR = 0.005
WEIGHT_DECAY = 5e-4
DROPOUT = 0.6
EPOCHS = 10_000
PATIENCE = 100


def train_epoch(model, data, optimizer):
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()


@torch.no_grad()
def evaluate_val_loss(model, data):
    model.eval()
    out = model(data.x, data.edge_index)
    return F.nll_loss(out[data.val_mask], data.y[data.val_mask]).item()


@torch.no_grad()
def test_accuracy(model, data):
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)
    return (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()


def run(
    dataset_name: str,
    seed: int = 42,
    data_parent: Optional[pathlib.Path] = None,
) -> float:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    parent = data_parent or PLANETOID_PARENT
    parent.mkdir(parents=True, exist_ok=True)
    root = str(parent / dataset_name)

    dataset = Planetoid(
        root=root,
        name=dataset_name,
        transform=NormalizeFeatures(),
    )
    data = dataset[0]
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    data = data.to(device)

    model = GAT(
        num_features=dataset.num_features,
        num_classes=dataset.num_classes,
        dropout=DROPOUT,
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    best_val_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())
    patience_counter = 0

    for _epoch in range(1, EPOCHS + 1):
        train_epoch(model, data, optimizer)
        val_loss = evaluate_val_loss(model, data)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
        if patience_counter >= PATIENCE:
            break

    model.load_state_dict(best_state)
    return test_accuracy(model, data)

In [5]:
# @title Multi-run evaluation (same as `code/evaluate.py`)
def run_many(dataset_name: str, num_runs: int) -> Tuple[float, float, pathlib.Path]:
    csv_path = RESULTS_DIR / f"{dataset_name}_results.csv"
    accuracies: List[float] = []
    with open(csv_path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["run", "test_accuracy"])
        for i in range(1, num_runs + 1):
            acc = run(dataset_name, seed=i)
            accuracies.append(acc)
            w.writerow([i, f"{acc:.6f}"])
            f.flush()
            sys.stdout.flush()
            print(f"Run {i:3d}/{num_runs}  test_acc={acc * 100:.2f}%", flush=True)
    mean_acc = float(np.mean(accuracies) * 100)
    std_acc = float(np.std(accuracies) * 100)
    print("\n" + "=" * 45)
    print(f"Dataset : {dataset_name}")
    print(f"Runs    : {num_runs}")
    print(f"Mean    : {mean_acc:.2f}%")
    print(f"Std     : {std_acc:.2f}%")
    print(f"Results saved to: {csv_path}")
    print("=" * 45)
    return mean_acc, std_acc, csv_path

In [6]:
# @title Run experiments
# Set NUM_RUNS=100 to match the paper; use 3–5 for a quick test.
NUM_RUNS = 100
DATASETS = ("Cora", "CiteSeer")
summary = {}
for name in DATASETS:
    m, s, _p = run_many(name, num_runs=NUM_RUNS)
    summary[name] = (m, s)
print("\nSummary (mean, std) %:")
for k, (m, s) in summary.items():
    print(f"  {k}: {m:.2f} ± {s:.2f}")

Processing...
Done!


Run   1/100  test_acc=82.90%
Run   2/100  test_acc=82.70%
Run   3/100  test_acc=83.90%
Run   4/100  test_acc=83.00%
Run   5/100  test_acc=82.80%
Run   6/100  test_acc=83.20%
Run   7/100  test_acc=83.10%
Run   8/100  test_acc=83.90%
Run   9/100  test_acc=83.40%
Run  10/100  test_acc=83.50%
Run  11/100  test_acc=83.30%
Run  12/100  test_acc=83.70%
Run  13/100  test_acc=83.20%
Run  14/100  test_acc=83.20%
Run  15/100  test_acc=83.30%
Run  16/100  test_acc=83.00%
Run  17/100  test_acc=82.90%
Run  18/100  test_acc=83.40%
Run  19/100  test_acc=83.50%
Run  20/100  test_acc=82.80%
Run  21/100  test_acc=83.00%
Run  22/100  test_acc=83.90%
Run  23/100  test_acc=83.00%
Run  24/100  test_acc=82.70%
Run  25/100  test_acc=83.00%
Run  26/100  test_acc=83.00%
Run  27/100  test_acc=83.60%
Run  28/100  test_acc=83.00%
Run  29/100  test_acc=83.50%
Run  30/100  test_acc=83.70%
Run  31/100  test_acc=83.70%
Run  32/100  test_acc=83.60%
Run  33/100  test_acc=83.00%
Run  34/100  test_acc=83.60%
Run  35/100  t

Processing...
Done!


Run   1/100  test_acc=70.70%
Run   2/100  test_acc=71.30%
Run   3/100  test_acc=70.70%
Run   4/100  test_acc=70.70%
Run   5/100  test_acc=71.60%
Run   6/100  test_acc=71.50%
Run   7/100  test_acc=71.00%
Run   8/100  test_acc=70.50%
Run   9/100  test_acc=71.10%
Run  10/100  test_acc=70.60%
Run  11/100  test_acc=70.60%
Run  12/100  test_acc=71.50%
Run  13/100  test_acc=70.70%
Run  14/100  test_acc=71.40%
Run  15/100  test_acc=71.50%
Run  16/100  test_acc=70.60%
Run  17/100  test_acc=70.30%
Run  18/100  test_acc=71.00%
Run  19/100  test_acc=71.20%
Run  20/100  test_acc=71.10%
Run  21/100  test_acc=71.00%
Run  22/100  test_acc=70.60%
Run  23/100  test_acc=70.70%
Run  24/100  test_acc=70.10%
Run  25/100  test_acc=71.30%
Run  26/100  test_acc=70.10%
Run  27/100  test_acc=72.40%
Run  28/100  test_acc=70.40%
Run  29/100  test_acc=70.70%
Run  30/100  test_acc=71.30%
Run  31/100  test_acc=71.20%
Run  32/100  test_acc=71.30%
Run  33/100  test_acc=71.20%
Run  34/100  test_acc=70.50%
Run  35/100  t